# Metricas do Projeto
## Importações

In [ ]:
import os

import pandas as pd
from matplotlib import pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import string
import seaborn as sns

## Definições

In [ ]:
name_map = {
    "tf_efficientnetv2_b1.in1k": "EfficientnetV2_B1",
    "tf_efficientnetv2_b0.in1k": "EfficientnetV2_B0",
}

In [ ]:
color_map = {
    "gflop"   : "#fc7f03",
    "params"  : "#fc0384",
    "f1_macro": "#D7477C",
    "f1_weighted": "#D75A47"
}

## Importando os resultados

In [ ]:
path = "../results.csv"
df = pd.read_csv(path)
df["model"] = df["model"].map(name_map).fillna(df["model"]) # Alterando o nome do modelo (apenas embelezamento dos gráficos)
df

## Calculando os GFLOPS

In [ ]:
def graphic_gflop(df, save=False):
    df_gflops = df.groupby("model")["gflops"].first().reset_index()

    # Tamanho e estrutura do gráfico
    fig, ax = plt.subplots(figsize=(7, 5))

    # Colocar valor em cima da barra
    bars = ax.bar(df_gflops["model"], df_gflops["gflops"], color=color_map["gflop"])
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, height, f"{height:.2f}", ha="center", va="bottom")

    # Colocar para ir até o máximo + 0.1 e ir de 0.1 em 0.1
    ax.set_ylim(top=df_gflops["gflops"].max() + 0.1)
    ax.yaxis.set_major_locator(ticker.MultipleLocator(0.1))

    # Texto
    ax.set_title("Complexidade Computacional por Arquitetura (GFLOPs)")
    ax.set_xlabel("Arquitetura")
    ax.set_ylabel("GFLOPs")
    plt.tight_layout()

    if save:
        os.makedirs("../figures", exist_ok=True)
        plt.savefig("../figures/gflops_por_arquitetura.pdf")
    plt.show()

graphic_gflop(df)

## Grafíco de Número de Parâmetros por Arquitetura

In [ ]:
def graphic_params(df, save=False):
    df_params = df.groupby("model")["num_params"].first().reset_index()
    df_params["num_params_m"] = df_params["num_params"] / 1e6

    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.bar(df_params["model"], df_params["num_params_m"], color=color_map["params"])

    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, height, f"{height:.2f}M", ha="center", va="bottom")

    ax.set_ylim(top=df_params["num_params_m"].max() + 0.5)
    ax.yaxis.set_major_locator(ticker.MultipleLocator(0.5))

    ax.set_title("Número de Parâmetros por Arquitetura")
    ax.set_xlabel("Arquitetura")
    ax.set_ylabel("Parâmetros (milhões)")
    plt.tight_layout()

    if save:
        os.makedirs("../figures", exist_ok=True)
        plt.savefig("../figures/params_por_arquitetura.pdf")
    plt.show()

graphic_params(df)

## Gráfico F1-Score por Arquitetura

In [ ]:
def make_labels(grouped):
    labels = []
    letters = string.ascii_uppercase

    for i, row in enumerate(grouped.itertuples()):
        letter = f"({letters[i]})"
        model = row.model
        mode = row.training_mode
        aug = "Com Aug" if row.augmentation else "Sem Aug"

        label = f"{letter}\n{model}\n{mode}\n{aug}"
        labels.append(label)

    return labels

In [ ]:
def graphic_f1(df, dataset_name, save=False):
    df_ds = df[df["dataset"] == dataset_name]

    # Média das 3 seeds
    grouped = df_ds.groupby(["model", "training_mode", "augmentation"])[["f1_macro_test", "f1_weighted_test"]].mean().reset_index()

    grouped["label"] = make_labels(grouped)

    x = np.arange(len(grouped))
    width = 0.35

    fig, ax = plt.subplots(figsize=(16, 6))
    bars_macro    = ax.bar(x - width/2, grouped["f1_macro_test"],   width, label="F1 Macro", color=color_map["f1_macro"])
    bars_weighted = ax.bar(x + width/2, grouped["f1_weighted_test"], width, label="F1 Weighted", color=color_map["f1_weighted"])

    # Valor em cima das barras
    for bar in bars_macro:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, height, f"{height:.2f}", ha="center", va="bottom", fontsize=7)

    for bar in bars_weighted:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, height, f"{height:.2f}", ha="center", va="bottom", fontsize=7)

    ax.set_ylim(0, 1.1)
    ax.yaxis.set_major_locator(ticker.MultipleLocator(0.1))

    ax.set_title(f"F1-Score por Configuração - {dataset_name}")
    ax.set_xlabel("Arquitetura / Modo / Augmentation")
    ax.set_ylabel("F1-Score (média das 3 seeds)")
    ax.set_xticks(x)
    ax.set_xticklabels(grouped["label"], fontsize=8, rotation=30, ha="right")
    ax.legend()
    plt.tight_layout()

    if save:
        os.makedirs("../figures", exist_ok=True)
        plt.savefig(f"../figures/f1_score_{dataset_name}.png")
    plt.show()

graphic_f1(df, "OralEpitheliumDB")
graphic_f1(df, "NDB-UFES")

In [ ]:
def create_stats_table(df, dataset_name):
    df_ds = df[df["dataset"] == dataset_name]

    grouped = (
        df_ds.groupby(["model", "training_mode", "augmentation"])
        .agg(
            val_acc_mean=("val_acc_best", "mean"),
            val_acc_std=("val_acc_best", "std"),
            acc_test_mean=("acc_test", "mean"),
            acc_test_std=("acc_test", "std"),
            f1_macro_mean=("f1_macro_test", "mean"),
            f1_macro_std=("f1_macro_test", "std"),
            f1_weighted_mean=("f1_weighted_test", "mean"),
            f1_weighted_std=("f1_weighted_test", "std"),
        )
        .reset_index()
    )

    grouped["augmentation"] = grouped["augmentation"].map({True: "Com Aug", False: "Sem Aug"})

    grouped = grouped.rename(
        columns={
            "model": "Modelo",
            "training_mode": "Modo",
            "augmentation": "Augmentation",
            "val_acc_mean": "Acurácia (média)",
            "val_acc_std": "Acurácia (std)",
            "acc_test_mean": "Acc Test (média)",
            "acc_test_std": "Acc Test (std)",
            "f1_macro_mean": "F1 Macro (média)",
            "f1_macro_std": "F1 Macro (std)",
            "f1_weighted_mean": "F1 Weighted (média)",
            "f1_weighted_std": "F1 Weighted (std)",
        }
    )

    return grouped.sort_values(["Modelo", "Modo", "Augmentation"]).reset_index(drop=True)


def save_stats_table(df, dataset_name, output_path):
    stats = create_stats_table(df, dataset_name)
    dirpath = os.path.dirname(output_path)
    if dirpath:
        os.makedirs(dirpath, exist_ok=True)
    stats.to_csv(output_path, index=False)
    return stats

# Tabelas para ambos os datasets
stats_oral = create_stats_table(df, "OralEpitheliumDB")
stats_ndb = create_stats_table(df, "NDB-UFES")

# Salvar em CSV
save_stats_table(df, "OralEpitheliumDB", "../figures/stats_OralEpitheliumDB.csv")
save_stats_table(df, "NDB-UFES", "../figures/stats_NDB-UFES.csv")

stats_oral, stats_ndb

In [ ]:
def graphic_accuracy(df, dataset_name, save=False):
    df_ds = df[df["dataset"] == dataset_name]

    # Média das 3 seeds (agora agrupando pela acurácia 'val_acc_best')
    grouped = df_ds.groupby(["model", "training_mode", "augmentation"])["val_acc_best"].mean().reset_index()

    # Usa a sua função auxiliar para gerar as legendas
    grouped["label"] = make_labels(grouped)

    x = np.arange(len(grouped))

    fig, ax = plt.subplots(figsize=(16, 6))

    # Cria as barras (apenas uma por configuração)
    bars_acc = ax.bar(x, grouped["val_acc_best"], width=0.6, label="Acurácia", color=color_map["f1_macro"])

    # Valor em cima das barras
    for bar in bars_acc:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, height, f"{height:.2f}", ha="center", va="bottom", fontsize=7)

    # Configurações dos eixos (indo de 0.1 em 0.1 até 1.1)
    ax.set_ylim(0, 1.1)
    ax.yaxis.set_major_locator(ticker.MultipleLocator(0.1))

    # Títulos e legendas ajustados para Acurácia
    ax.set_title(f"Acurácia por Configuração - {dataset_name}")
    ax.set_xlabel("Arquitetura / Modo / Augmentation")
    ax.set_ylabel("Acurácia (média das 3 seeds)")

    ax.set_xticks(x)
    # Incluído a rotação de 45 graus para garantir que a leitura fique boa
    ax.set_xticklabels(grouped["label"], fontsize=8, rotation=45, ha="right")

    ax.legend()
    plt.tight_layout()

    if save:
        os.makedirs("../figures", exist_ok=True)
        plt.savefig(f"../figures/accuracy_{dataset_name}.png")
    plt.show()

# Chamadas da nova função
graphic_accuracy(df, "OralEpitheliumDB")
graphic_accuracy(df, "NDB-UFES")

## Matrizes de Confusão

In [ ]:
import json
import seaborn as sns

def graphic_confusion_matrix(dataset_name, model_name, training_mode, augmentation, seed, save=True):
    """
    Gera e exibe a matriz de confusão para uma execução específica.
    
    Parameters:
    - dataset_name: Nome do dataset (ex: "OralEpitheliumDB", "NDB-UFES")
    - model_name: Nome do modelo (ex: "tf_efficientnetv2_b0.in1k")
    - training_mode: Modo de treinamento (ex: "scratch", "feature_extraction", "fine_tuning")
    - augmentation: Boolean indicando se houve augmentation
    - seed: Seed utilizado
    - save: Se True, salva em PDF vetorial
    """
    
    # Construir o nome do arquivo
    name = f"{dataset_name}_{model_name}_{training_mode}_aug{augmentation}_seed{seed}"
    confusion_matrix_path = f"../confusion_matrices/{name}.json"
    
    try:
        with open(confusion_matrix_path, 'r') as f:
            cm = json.load(f)
        
        # Converter para numpy array
        cm = np.array(cm)
        
        # Criar figura
        fig, ax = plt.subplots(figsize=(8, 6))
        
        # Criar heatmap
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar_kws={'label': 'Contagem'})
        
        # Configurações
        model_display = name_map.get(model_name, model_name)
        aug_text = "Com Aug" if augmentation else "Sem Aug"
        
        ax.set_title(f"Matriz de Confusão - {dataset_name}\n{model_display} | {training_mode} | {aug_text} | Seed: {seed}")
        ax.set_ylabel("Verdadeiro")
        ax.set_xlabel("Predito")
        
        plt.tight_layout()
        
        if save:
            os.makedirs("../figures", exist_ok=True)
            pdf_path = f"../figures/confusion_matrix_{name}.pdf"
            plt.savefig(pdf_path, format='pdf')
            print(f"Salvo: {pdf_path}")
        
        plt.show()
        
    except FileNotFoundError:
        print(f"Arquivo não encontrado: {confusion_matrix_path}")

# Exemplo de uso:
# graphic_confusion_matrix("NDB-UFES", "tf_efficientnetv2_b0.in1k", "fine_tuning", True, 42)

## Curvas de Aprendizado

In [ ]:
def graphic_learning_curves(dataset_name, model_name, training_mode, augmentation, seed, save=True):
    """
    Gera e exibe as curvas de aprendizado (loss e acurácia) para uma execução específica.
    
    Parameters:
    - dataset_name: Nome do dataset (ex: "OralEpitheliumDB", "NDB-UFES")
    - model_name: Nome do modelo (ex: "tf_efficientnetv2_b0.in1k")
    - training_mode: Modo de treinamento (ex: "scratch", "feature_extraction", "fine_tuning")
    - augmentation: Boolean indicando se houve augmentation
    - seed: Seed utilizado
    - save: Se True, salva em PDF vetorial
    """
    
    # Construir o nome do arquivo
    name = f"{dataset_name}_{model_name}_{training_mode}_aug{augmentation}_seed{seed}"
    history_path = f"../histories/{name}.json"
    
    try:
        with open(history_path, 'r') as f:
            history = json.load(f)
        
        # Criar figura com 2 subplots (loss e acurácia)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        epochs = range(1, len(history['train_loss']) + 1)
        
        # Plot de Loss
        ax1.plot(epochs, history['train_loss'], 'b-', label='Treino', linewidth=2)
        ax1.plot(epochs, history['val_loss'], 'r-', label='Validação', linewidth=2)
        ax1.set_xlabel('Época')
        ax1.set_ylabel('Loss')
        ax1.set_title('Curva de Aprendizado - Loss')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Plot de Acurácia
        ax2.plot(epochs, history['train_acc'], 'b-', label='Treino', linewidth=2)
        ax2.plot(epochs, history['val_acc'], 'r-', label='Validação', linewidth=2)
        ax2.set_xlabel('Época')
        ax2.set_ylabel('Acurácia')
        ax2.set_title('Curva de Aprendizado - Acurácia')
        ax2.set_ylim([0, 1])
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # Título geral
        model_display = name_map.get(model_name, model_name)
        aug_text = "Com Aug" if augmentation else "Sem Aug"
        fig.suptitle(f"Curvas de Aprendizado - {dataset_name}\n{model_display} | {training_mode} | {aug_text} | Seed: {seed}", 
                     fontsize=12, fontweight='bold')
        
        plt.tight_layout()
        
        if save:
            os.makedirs("../figures", exist_ok=True)
            pdf_path = f"../figures/learning_curves_{name}.pdf"
            plt.savefig(pdf_path, format='pdf')
            print(f"Salvo: {pdf_path}")
        
        plt.show()
        
    except FileNotFoundError:
        print(f"Arquivo não encontrado: {history_path}")

# Exemplo de uso:
# graphic_learning_curves("NDB-UFES", "tf_efficientnetv2_b0.in1k", "fine_tuning", True, 42)

## Gerar Todos os Gráficos

In [ ]:
# Gerar todas as matrizes de confusão e curvas de aprendizado
import glob
import re

# Listar todos os arquivos de histórico
history_files = glob.glob("../histories/*.json")

print(f"Gerando {len(history_files)} matrizes de confusão e curvas de aprendizado...")
print("=" * 80)

for history_file in sorted(history_files):
    # Extrair informações do nome do arquivo
    filename = os.path.basename(history_file)[:-5]  # Remove .json
    
    # Parsing usando regex
    # Padrão: {dataset}_{model}_{training_mode}_{augmentation}_{seed}
    # Exemplo: NDB-UFES_tf_efficientnetv2_b0.in1k_feature_extraction_augFalse_seed42
    
    match = re.match(
        r"^(OralEpitheliumDB|NDB-UFES)_(.+?)_(scratch|feature_extraction|fine_tuning)_(aug(True|False))_seed(\d+)$",
        filename
    )
    
    if not match:
        print(f"⚠ Erro ao processar: {filename}")
        continue
    
    dataset_name = match.group(1)
    model_name = match.group(2)
    training_mode = match.group(3)
    augmentation = match.group(5) == "True"
    seed = int(match.group(6))
    
    print(f"✓ {dataset_name} | {model_name} | {training_mode} | Aug: {augmentation} | Seed: {seed}")
    
    # Gerar gráficos
    try:
        graphic_learning_curves(dataset_name, model_name, training_mode, augmentation, seed, save=True)
        graphic_confusion_matrix(dataset_name, model_name, training_mode, augmentation, seed, save=True)
    except Exception as e:
        print(f"  ✗ Erro ao gerar gráficos: {e}")

print("\n" + "=" * 80)
print("✓ Todos os gráficos foram gerados e salvos em PDF vetorial!")
print(f"✓ Pasta 'figures/' contém todos os arquivos PDF")